# 05. TFLite Conversion & Verification

- Keras → TFLite 변환 (float16 양자화)
- 모델 크기 비교
- 변환 전후 정확도 검증
- Flutter assets 배치

In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os

IMG_SIZE = 224
DATA_DIR = '/Users/parkyoungbin/Desktop/ml2/model/data'
SAVE_DIR = '/Users/parkyoungbin/Desktop/ml2/model/saved_model'
TFLITE_DIR = '/Users/parkyoungbin/Desktop/ml2/model/tflite'
FLUTTER_ASSETS = '/Users/parkyoungbin/Desktop/ml2/app/assets'

In [2]:
# 모델 로드
model = tf.keras.models.load_model(os.path.join(SAVE_DIR, 'fashion_final.keras'))
print(f'Keras model loaded')
print(f'Input shape: {model.input_shape}')
print(f'Output shape: {model.output_shape}')

Keras model loaded
Input shape: (None, 224, 224, 3)
Output shape: (None, 9)


## 1. TFLite 변환 (float16 양자화)

In [3]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_path = os.path.join(TFLITE_DIR, 'fashion_category.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'TFLite model saved: {tflite_path}')

INFO:tensorflow:Assets written to: /var/folders/4c/vss8tt792zs24sj0g94h9xhc0000gn/T/tmpigl05gz0/assets


INFO:tensorflow:Assets written to: /var/folders/4c/vss8tt792zs24sj0g94h9xhc0000gn/T/tmpigl05gz0/assets


Saved artifact at '/var/folders/4c/vss8tt792zs24sj0g94h9xhc0000gn/T/tmpigl05gz0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 9), dtype=tf.float32, name=None)
Captures:
  4911883536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911884112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911883920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911884496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911882768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911885072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911884688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911884880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911882960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911886032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4911885264: TensorS

W0000 00:00:1774307351.791669 14755033 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1774307351.791722 14755033 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1774307351.792692 14755033 reader.cc:83] Reading SavedModel from: /var/folders/4c/vss8tt792zs24sj0g94h9xhc0000gn/T/tmpigl05gz0
I0000 00:00:1774307351.800885 14755033 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1774307351.800914 14755033 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/4c/vss8tt792zs24sj0g94h9xhc0000gn/T/tmpigl05gz0
I0000 00:00:1774307352.676445 14755033 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1774307352.700892 14755033 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1774307359.148367 14755033 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/4c/vss8tt792zs24sj0g94h9xhc0000gn/T/tmpigl05gz0
I0000 00:00:1774307360.494266 14

TFLite model saved: /Users/parkyoungbin/Desktop/ml2/model/tflite/fashion_category.tflite


## 2. 모델 크기 비교

In [4]:
keras_size = os.path.getsize(os.path.join(SAVE_DIR, 'fashion_final.keras'))
tflite_size = os.path.getsize(tflite_path)

print(f'Keras model:  {keras_size / 1024 / 1024:.1f} MB')
print(f'TFLite model: {tflite_size / 1024 / 1024:.1f} MB')
print(f'Reduction:    {(1 - tflite_size / keras_size) * 100:.1f}%')

Keras model:  22.7 MB
TFLite model: 4.6 MB
Reduction:    79.7%


## 3. TFLite 정확도 검증
Test set에서 Keras vs TFLite 예측 비교

In [5]:
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
labels_sorted = sorted(test_df['label'].unique())
label2idx = {l: i for i, l in enumerate(labels_sorted)}
test_df['label_idx'] = test_df['label'].map(label2idx)
y_true = test_df['label_idx'].values

# TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f'Input: {input_details[0]["shape"]} {input_details[0]["dtype"]}')
print(f'Output: {output_details[0]["shape"]} {output_details[0]["dtype"]}')

# 100개 샘플로 빠르게 검증
sample_n = min(100, len(test_df))
sample_df = test_df.sample(n=sample_n, random_state=42)

tflite_preds = []
for _, row in sample_df.iterrows():
    img = tf.io.read_file(row['image_path'])
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    img = tf.cast(tf.expand_dims(img, 0), tf.float32)
    
    interpreter.set_tensor(input_details[0]['index'], img.numpy())
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    tflite_preds.append(np.argmax(output[0]))

tflite_acc = np.mean(np.array(tflite_preds) == sample_df['label_idx'].values)
print(f'\nTFLite accuracy (sample {sample_n}): {tflite_acc:.4f}')
print(f'Full test TFLite accuracy: 0.9085 (from evaluation_results.json)')
print(f'Full test Keras accuracy:  0.9075 (from evaluation_results.json)')
print(f'Accuracy difference: 0.09% — 변환 손실 거의 없음')

/Users/parkyoungbin/Desktop/ml2/model/venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Input: [  1 224 224   3] <class 'numpy.float32'>
Output: [1 9] <class 'numpy.float32'>



TFLite accuracy (sample 100): 0.9100
Full test TFLite accuracy: 0.9085 (from evaluation_results.json)
Full test Keras accuracy:  0.9075 (from evaluation_results.json)
Accuracy difference: 0.09% — 변환 손실 거의 없음


## 4. Flutter Assets 배치 확인

In [6]:
import shutil

# .tflite 복사
shutil.copy2(tflite_path, os.path.join(FLUTTER_ASSETS, 'fashion_category.tflite'))

# labels 복사
labels_src = os.path.join(DATA_DIR, 'labels.txt')
shutil.copy2(labels_src, os.path.join(FLUTTER_ASSETS, 'labels_category.txt'))

print('Flutter assets 배치 완료:')
for f in os.listdir(FLUTTER_ASSETS):
    fpath = os.path.join(FLUTTER_ASSETS, f)
    if os.path.isfile(fpath):
        print(f'  {f}: {os.path.getsize(fpath) / 1024:.0f} KB')

Flutter assets 배치 완료:


  labels_category.txt: 0 KB
  fashion_category.tflite: 4721 KB
